# Mode C: Full experiment

Runs every stage for both models, then writes the results into the paper. Each stage saves its output to Google Drive and skips work already done, so if Colab disconnects, reconnect and run the cells again from the top. Stages run one cell at a time so a session limit only loses the current stage.

## 1. Settings
Edit `REPO_URL` once the repository is on GitHub. Keep `USE_DRIVE = True` so results survive a disconnect.

In [ ]:
MODE = "full"
CONFIG = "configs/full.yaml"
REPO_URL = "https://github.com/YOUR-ACCOUNT/agreement-is-not-truth.git"  # or upload the ZIP to Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/agreement-is-not-truth.zip"     # used when REPO_URL is not set
USE_DRIVE = True
INSTALL_VLLM = False  # True on L4/A100 for speed; the code falls back to transformers if vLLM fails
PROJECT = "/content/agreement-is-not-truth"

## 2. Hardware check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 3. Get the code and connect Drive

In [ ]:
import os, subprocess
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.exists(PROJECT):
    if 'YOUR-ACCOUNT' not in REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, PROJECT], check=True)
    else:
        subprocess.run(['unzip', '-q', ZIP_ON_DRIVE, '-d', '/content'], check=True)
os.chdir(PROJECT)
if USE_DRIVE:
    store = '/content/drive/MyDrive/agreement-is-not-truth-results'
    os.makedirs(store, exist_ok=True)
    if not os.path.islink('results'):
        subprocess.run(['rm', '-rf', 'results'])
        os.symlink(store, 'results')
print(os.getcwd(), os.listdir('.'))

Mounted at /content/drive
/content/agreement-is-not-truth ['pyproject.toml', 'README.md', 'LICENSE', 'src', 'tools', 'docs', 'requirements.txt', 'notebooks', 'scripts', 'plans', 'tests', 'configs', 'run', 'results', 'CODE_GUIDE.md', 'paper', '.gitignore']


## 4. Install dependencies

In [ ]:
!pip install -q -e .[gpu,judge,dev]
if INSTALL_VLLM:
    !pip install -q vllm

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pressure-probe (pyproject.toml) ... done


## 5. Keys
Add `HF_TOKEN` and (optional) `OPENAI_API_KEY` in Colab's Secrets panel (key icon on the left). Without an OpenAI key the judge falls back to alias matching plus NLI, and the paper reports which judge was used.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less GPU memory fragmentation
try:
    from google.colab import userdata
    for name in ('HF_TOKEN', 'OPENAI_API_KEY'):
        try:
            os.environ[name] = userdata.get(name)
            print(name, 'loaded')
        except Exception:
            print(name, 'not set')
except ImportError:
    print('Not on Colab; set keys in the environment')

HF_TOKEN loaded
OPENAI_API_KEY not set


## 7. Run the experiment
Each cell is one stage. Rerunning a cell skips finished work, so it is safe after a disconnect.

In [ ]:
!python -m pressure_probe.cli stage prepare --config $CONFIG

2026-10-28 04:46:49,827 INFO pressure_probe.stages.prepare: Environment: {'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'gpu': 'Tesla T4', 'gpu_memory_gb': 14.6}
2026-10-28 04:46:50,256 INFO pressure_probe.data: Questions already prepared at results/full/questions.jsonl


In [ ]:
!python -m pressure_probe.cli stage generate --config $CONFIG

2026-10-28 04:46:59,964 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage generate --config configs/full.yaml --model qwen7b
2026-10-28 04:47:02,049 INFO pressure_probe.pipeline: === generate [qwen7b] ===
2026-10-28 04:47:03,563 INFO pressure_probe.stages.generate: [qwen7b] generation already complete
2026-10-28 04:47:03,747 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage generate --config configs/full.yaml --model llama8b
2026-10-28 04:47:04,969 INFO pressure_probe.pipeline: === generate [llama8b] ===
2026-10-28 04:47:05,557 INFO pressure_probe.stages.generate: [llama8b] generation already complete


In [ ]:
!python -m pressure_probe.cli stage judge --config $CONFIG

2026-10-28 04:47:09,190 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage judge --config configs/full.yaml --model qwen7b
2026-10-28 04:47:10,420 INFO pressure_probe.pipeline: === judge [qwen7b] ===
2026-10-28 04:47:10,762 INFO pressure_probe.stages.checks: [qwen7b] judge already complete
2026-10-28 04:47:10,935 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage judge --config configs/full.yaml --model llama8b
2026-10-28 04:47:12,116 INFO pressure_probe.pipeline: === judge [llama8b] ===
2026-10-28 04:47:12,442 INFO pressure_probe.stages.checks: [llama8b] judge already complete


In [ ]:
!python -m pressure_probe.cli stage sample_scores --config $CONFIG

2026-10-28 04:47:25,910 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage sample_scores --config configs/full.yaml --model qwen7b
2026-10-28 04:47:27,394 INFO pressure_probe.pipeline: === sample_scores [qwen7b] ===
2026-10-28 04:47:27,772 INFO pressure_probe.stages.checks: [qwen7b] sample_scores already complete
2026-10-28 04:47:27,940 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage sample_scores --config configs/full.yaml --model llama8b
2026-10-28 04:47:29,120 INFO pressure_probe.pipeline: === sample_scores [llama8b] ===
2026-10-28 04:47:29,527 INFO pressure_probe.stages.checks: [llama8b] sample_scores already complete


In [ ]:
!python -m pressure_probe.cli stage split --config $CONFIG

2026-10-28 04:47:35,297 INFO pressure_probe.pipeline: === split [qwen7b] ===
2026-10-28 04:47:35,393 INFO pressure_probe.stages.split: [qwen7b] split counts {'CW': 315, 'CR': 374, 'INC': 425, 'EXCL': 3}; matched set 240
2026-10-28 04:47:35,394 INFO pressure_probe.pipeline: === split [llama8b] ===
2026-10-28 04:47:35,466 INFO pressure_probe.stages.split: [llama8b] split counts {'CW': 195, 'CR': 309, 'INC': 607, 'EXCL': 6}; matched set 230


In [ ]:
!python -m pressure_probe.cli stage probe --config $CONFIG

2026-10-28 04:48:10,676 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage probe --config configs/full.yaml --model qwen7b
2026-10-28 04:48:11,849 INFO pressure_probe.pipeline: === probe [qwen7b] ===
2026-10-28 04:48:12,241 INFO pressure_probe.stages.probe: [qwen7b] probes already complete
2026-10-28 04:48:12,406 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage probe --config configs/full.yaml --model llama8b
2026-10-28 04:48:13,563 INFO pressure_probe.pipeline: === probe [llama8b] ===
2026-10-28 04:48:15,654 WARNING pressure_probe.backends: vLLM unavailable (No module named 'vllm'); falling back to transformers
2026-10-28 04:48:25,698 INFO httpx: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-28 04:48:25,792 INFO httpx: HTTP Request: HEAD https://huggingface.co/unsloth/Llama-3.1-8B-Instruct/resolve/4699cc75b550f9c6f3173fb80f4703b62d946aa5/config.json "HTTP/1.1 307 Temporary Redire

In [ ]:
!python -m pressure_probe.cli stage metaqa --config $CONFIG

2026-10-28 05:08:24,007 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage metaqa --config configs/full.yaml --model qwen7b
2026-10-28 05:08:25,184 INFO pressure_probe.pipeline: === metaqa [qwen7b] ===
2026-10-28 05:08:27,226 WARNING pressure_probe.backends: vLLM unavailable (No module named 'vllm'); falling back to transformers
2026-10-28 05:08:31,720 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-28 05:08:31,803 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json?%2FQwen%2FQwen2.5-7B-Instruct%2Fresolve%2Fa09a35458c702b33eeacc393d103063234e8bc28%2Fconfig.json=&etag=%220178295f88afc3c7f279ed284f961f8c1be00654%22 "HTTP/1.1 200 OK"
2026-10-28 05:08:31,889 INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-c

In [ ]:
!python -m pressure_probe.cli stage probe_scores --config $CONFIG

2026-10-28 07:29:23,181 INFO pressure_probe.pipeline: Running /usr/bin/python3 -m pressure_probe.cli stage probe_scores --config configs/full.yaml --model qwen7b
2026-10-28 07:29:24,445 INFO pressure_probe.pipeline: === probe_scores [qwen7b] ===
2026-10-28 07:29:24,475 INFO pressure_probe.nli: Loading NLI model microsoft/deberta-large-mnli
2026-10-28 07:29:30,922 INFO httpx: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-large-mnli/resolve/7296194b9009373def4f7c5dad292651e4b5cf4e/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-28 07:29:30,961 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-large-mnli/7296194b9009373def4f7c5dad292651e4b5cf4e/config.json?%2Fmicrosoft%2Fdeberta-large-mnli%2Fresolve%2F7296194b9009373def4f7c5dad292651e4b5cf4e%2Fconfig.json=&etag=%226c9d2ed40c51a61de316577d0defa29b2c6c2c3b%22 "HTTP/1.1 200 OK"
2026-10-28 07:29:31,006 INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/mode

In [ ]:
!python -m pressure_probe.cli stage evaluate --config $CONFIG

2026-10-28 07:33:09,209 INFO pressure_probe.stages.evaluate: Wrote results/full/summary.json


In [ ]:
import json
s = json.load(open('results/full/summary.json'))
for key, m in s['models'].items():
    print('\n' + m['display_name'], '| CW', m['counts']['CW'], '| matched', m['matched'])
    for name, meth in m['methods'].items():
        a = meth['auroc_matched']
        print(f"   {name:18s} {a['value']:.3f} [{a['low']:.2f}, {a['high']:.2f}]")
    print('   hypotheses:', {h: v.get('supported') for h, v in m['hypotheses'].items()})


Qwen2.5-7B-Instruct | CW 315 | matched {'n': 240, 'CW': 120, 'CR': 120}
   logprob            0.576 [0.50, 0.65]
   ptrue              0.552 [0.48, 0.62]
   verbal             0.593 [0.52, 0.66]
   semantic_entropy   0.533 [0.49, 0.58]
   selfcheck_nli      0.547 [0.49, 0.61]
   cross_family       0.645 [0.58, 0.71]
   metaqa             0.702 [0.63, 0.77]
   dfr                0.612 [0.55, 0.68]
   dfr_cross          0.670 [0.60, 0.74]
   f_counter          0.608 [0.54, 0.67]
   cps                0.518 [0.44, 0.59]
   hypotheses: {'H1': True, 'H2': False, 'H3': False, 'H4': True}

Llama-3.1-8B-Instruct | CW 195 | matched {'n': 230, 'CW': 120, 'CR': 110}
   logprob            0.633 [0.56, 0.70]
   ptrue              0.536 [0.46, 0.61]
   verbal             0.597 [0.53, 0.66]
   semantic_entropy   0.508 [0.46, 0.56]
   selfcheck_nli      0.533 [0.47, 0.60]
   cross_family       0.686 [0.62, 0.75]
   metaqa             0.696 [0.63, 0.76]
   dfr                0.544 [0.48, 0.61]
   dfr_

In [ ]:
import json
s = json.load(open('results/full/summary.json'))
for key, m in s['models'].items():
    print('\n' + m['display_name'])
    for b, c in m['comparisons_vs_ours'].items():
        d = c['difference']
        print(f"   DFR vs {b:16s} diff {d['value']:+.3f} [{d['low']:+.2f}, {d['high']:+.2f}]  p_holm {c['p_holm']:.3f}")
    print('   H2:', m['hypotheses']['H2'])
    print('   H3:', m['hypotheses']['H3'])
    print('   counts:', m['counts']['per_dataset'])


Qwen2.5-7B-Instruct
   DFR vs logprob          diff +0.036 [-0.06, +0.13]  p_holm 1.000
   DFR vs ptrue            diff +0.060 [-0.03, +0.15]  p_holm 0.736
   DFR vs verbal           diff +0.019 [-0.06, +0.10]  p_holm 1.000
   DFR vs semantic_entropy diff +0.080 [+0.00, +0.16]  p_holm 0.251
   DFR vs selfcheck_nli    diff +0.065 [-0.02, +0.15]  p_holm 0.628
   DFR vs cross_family     diff -0.033 [-0.12, +0.06]  p_holm 1.000
   DFR vs metaqa           diff -0.090 [-0.18, -0.00]  p_holm 0.256
   H2: {'dfr': 0.6122222222222222, 'dfr_low': 0.5450624999999999, 'sampling': {'semantic_entropy': {'auroc': 0.5326041666666667, 'p_holm': 0.2505442044428313}, 'selfcheck_nli': {'auroc': 0.5473958333333333, 'p_holm': 0.6275407924842522}}, 'supported': False, 'best_text_baseline': 'metaqa', 'baseline_auroc': 0.7024305555555556, 'baseline_p_holm': 0.25629623580146277}
   H3: {'dfr': 0.6122222222222222, 'cps': 0.5179166666666666, 'difference': {'value': 0.09430555555555564, 'low': -0.02564062499999989

## 8. Human labels (needed for the paper)
Creates `human_labels.csv` with 100 answers. Download it, have two authors fill `rater1` and `rater2` independently (1 = correct, 0 = wrong), and put the filled file back in the same folder. The report then prints Cohen's kappa in the paper.

In [ ]:
!python scripts/make_label_sheet.py --config $CONFIG

Wrote 100 rows to results/full/human_labels.csv. Judge labels are kept out of the sheet on purpose.
Judge method counts for reference: {'refusal': 3, 'nli': 853, 'alias': 261}


## 9. Install LaTeX and pandoc (about 5 minutes)

In [ ]:
!apt-get -qq update && apt-get -qq install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-science latexmk > /dev/null && echo LaTeX done
# Colab's own pandoc is too old for the Word export, so install a current release.
!wget -q https://github.com/jgm/pandoc/releases/download/3.1.3/pandoc-3.1.3-1-amd64.deb -O /tmp/pandoc.deb && dpkg -i /tmp/pandoc.deb > /dev/null && pandoc --version | head -1

## 10. Write results into the paper
Fills tables, figures and numbers, compiles the IEEE PDF, writes the Word file, checks the page count and builds the ZIP.

In [ ]:
!python -m pressure_probe.cli report --config $CONFIG

In [ ]:
import json
report = json.load(open('paper/validation_report.json'))
print(json.dumps({k: v for k, v in report.items() if k != 'unresolved_tokens'}, indent=1))
print('Unresolved tokens:', report['unresolved_tokens'])

## 11. Download

In [ ]:
from google.colab import files
files.download(f'dist/{MODE}_package.zip')
files.download('paper/paper.pdf')